[\[GitHub\] Jupyter Notebook](https://github.com/sslastochkin/mcab/blob/main/docs/guide/01_why_monte_carlo.ipynb)
  
# Why Monte Carlo Over Formulas  
  
You want to run an A/B test.  
The very first question is — **how many users do you need** and **what effect size is actually detectable**.  
You open any online sample size calculator, enter three numbers, and get an answer in half a second. For free.  
  
So why would you need some simulation library that computes the same numbers
in minutes instead of milliseconds?
  
Short answer: **a calculator always gives you an answer — but not always the right one.**  
It does not know what *your* data looks like, and silently substitutes assumptions for you —  
assumptions that are often violated for real product metrics.  
  
In this lesson we will:  
1. Understand what exactly an online calculator computes and what formulas it uses.
2. Write out those formulas explicitly and honestly ask: can we trust them?
3. Show that on "well-behaved" data `mcab` produces **exactly the same** `n` and `MDE`
   as the formula — meaning Monte Carlo does not disagree with the math.
4. Explain what Monte Carlo is and **why** it can be trusted.
5. Demonstrate real cases where the formula is wrong but Monte Carlo is not, and prove why.

## 1. What Online Calculators Do

A sample size calculator answers one of two symmetric questions:

- **Sample size:** "What effect `MDE` do I want to detect? → how many observations `n` do I need?"
- **MDE (Minimal Detectable Effect):** "I have `n` observations → what is the minimum effect I can actually detect?"

Both answers depend on four quantities:

| Notation | Meaning | Who sets it |
|---|---|---|
| $\alpha$ | probability of a false positive (usually 0.05) | you |
| $\beta$ | probability of missing the effect; power = $1 - \beta$ (usually 0.8) | you |
| $\sigma$ | spread (standard deviation) of the metric | data |
| $\Delta$ | effect size (MDE) | you / data |

### Key Points to Pay Attention To  

The calculator takes $\Delta$ (MDE) as a shift in the mean — it assumes that the test group differs from the control group on average by exactly $\Delta$, while the spread $\sigma$ remains unchanged.  

This works if the effect is additive: each user gets a fixed constant added (or at least the average of the additions equals $\Delta$ and the additions do not depend on the original value).  

If the effect is multiplicative (each user grows by some % of their previous value) — users with high revenue get a larger addition → variance in the test group increases → the formula underestimates the required n.  
  
Let us look in more detail at how this affects Type I and Type II errors.  
  
#### Type I Error $\alpha$  
  
> Type I error — the probability of detecting an effect where there actually is none (false positive — we say the effect exists when it does not).  
  
*✅ A multiplicative effect does not break $\alpha$ and does not increase the Type I error rate.*  
  
$\alpha$ is measured under $H_0$: both groups are drawn from the same distribution (no effect).  
In this world both groups have the same $\sigma$ — there is no asymmetric variance inflation.  
The test is correctly calibrated, 5% false positives remain 5%.  
  
#### Type II Error $\beta$  
  
> Type II error — the probability of missing an effect that truly exists (when the effect is present we say it is absent)
  
*❌ With a multiplicative effect the actual Type II error will be larger than what we plugged into the formula! Let us see why.*  
  
The formula assumes the following standard error:  
  
$$
\text{SE} = \sqrt{\dfrac{2 \sigma^2}{n}}
$$  
  
But under a multiplicative effect the real standard error is larger because multiplicativity inflates variance:  
  
$$
\widehat{\text{SE}} = \sqrt{\dfrac{\sigma^2}{n} + \dfrac{\sigma^2 (1 + \epsilon)^2}{n}}
$$  
  
**What are the consequences?**  
The test statistic is smaller than the formula expected → the effect is detected less often → power drops from 80% to ~65% → $\beta$ grows from 20% to ~35%.  
  
**Example to illustrate**  
```
Formula:  t_expected = Δ / SE_formula = 30 / 10.69 ≈ 2.81 -> power ≈ 80%
Reality:  t_real     = Δ / SE_real    = 30 / 12.40 ≈ 2.42 -> power ≈ 68%
```
  
**Practical implication**  
An analyst will launch an experiment confident in 80% power. The test will not show a significant result — the analyst will say "no effect" or "we need more data". In reality the effect exists; the experiment was simply underpowered. The formula gave a false sense of confidence — not in the direction of false discoveries, but in the direction of missed ones.  
  
#### Choice of Test Statistic  
  
The formulas for $n$ and $\text{MDE}$ are derived for testing the difference in means.  
But what if we care not about the mean but, say, the median?  
The calculator formula will no longer work.  
  
#### Choice of Statistical Test  
  
The formulas for $n$ and $\text{MDE}$ work only when the test statistic under $H_1$ has a known distribution with an analytically computable non-centrality parameter $\lambda(\Delta, \sigma, n)$. This allows inverting the problem: set the desired power → find $n$.

This does not work for three classes of tests:

1. Resampling tests (bootstrap, permutation test)

   Their p-value is determined by the empirical distribution of the statistic built from the data itself. There is no analytical formula for the distribution — it depends on the entire shape of the data, not just $\sigma$ and $\Delta$.

2. Ratio metrics (delta method, CTR)

   The variance of the ratio $\dfrac{\mu_{\text{num}}}{\mu_{\text{denom}}}$ depends on cov(num, denom) and both second moments.  

3. Rank-based and non-parametric tests (Mann–Whitney, etc.)

   Power depends not on $\mu$ and $\sigma$, but on the full shape of the distribution.   

One sentence: the formula works where the test statistic is asymptotically normal and its variance is expressed through $\sigma$ and $n$ analytically.

## 2. The Formulas Hidden Behind the "Calculate" Button

Almost all calculators for comparing two means (two equal groups,
two-sided test) use the following.

**Sample size per group:**

$$
n = \frac{2\,\sigma^2\,\left(z_{1-\alpha/2} + z_{1-\beta}\right)^2}{\Delta^2}
$$

**Minimum detectable effect for a given `n` per group:**

$$
\Delta = \left(z_{1-\alpha/2} + z_{1-\beta}\right)\,\sigma\,\sqrt{\frac{2}{n}}
$$

where $z_{1-\alpha/2}$ and $z_{1-\beta}$ are quantiles of the standard normal distribution.

For conversion rates (proportions) the formula has the same structure, only $\sigma^2$
is replaced by $p(1-p)$:

$$
n = \frac{\left(z_{1-\alpha/2} + z_{1-\beta}\right)^2 \, 2\,\bar p (1-\bar p)}{\Delta^2}
$$

Looks solid. But notice what it all relies on. The formulas are valid
**only if** all of the following hold simultaneously:

1. **Normality** of the mean estimate (or a good enough approximation via CLT).
2. **Independence** of observations from each other.
3. **Known and constant** variance `σ`.
4. You are analyzing **exactly that mean** for which the formula was derived
   (not a ratio, not a median, not the 95th percentile).

Something to think about 🤔: how often do **all four** conditions hold simultaneously for your real metrics? Revenue per user (lots of zeros + rare "whales"), CTR
(one user = many impressions → observations are dependent), time on site (heavy tail) —
nearly every assumption breaks here.
  
Formulas in code:

In [1]:
import numpy as np
from scipy import stats
from scipy.stats import norm

def mde_formula(std, n_per_group, alpha=0.05, power=0.8):
    """MDE via the classical formula for the difference of two means."""
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    return z * std * np.sqrt(2 / n_per_group)

def n_per_group_formula(std, mde, alpha=0.05, power=0.8):
    """Sample size per group via the classical formula."""
    z = norm.ppf(1 - alpha / 2) + norm.ppf(power)
    return int(np.ceil(2 * std**2 * z**2 / mde**2))

# Example: metric with mean=100 and std=30, two groups of 10,000 each
STD = 30
N_PER_GROUP = 10_000

mde_theory = mde_formula(STD, N_PER_GROUP)
print("Examples.")
print(f"Formula: with n={N_PER_GROUP} per group the minimum detectable effect MDE ≈ {mde_theory:.4f}")
print(f"Formula: to detect this effect you need n ≈ "
      f"{n_per_group_formula(STD, mde_theory)} per group")


Examples.
Formula: with n=10000 per group the minimum detectable effect MDE ≈ 1.1886
Formula: to detect this effect you need n ≈ 10000 per group


## 3. Trust Check: Does `mcab` Actually Agree with the Formula?

Before criticizing formulas, the honest move is to verify that the new tool
is **at least as good** as the old one where the old one is known to be correct.

Formulas are honest in exactly one ideal world: when data are normal and independent.
Let us generate exactly such data and ask `mcab` to compute `MDE` and `n`
**via simulation**, knowing nothing about the formula. If Monte Carlo returns the same numbers (with small simulation noise) — it can be trusted at least as much as the formulas.  

In [2]:
from mcab import RandomData, AaDataIid, DesignerIid

# 1. Generate data that is ideal for the formula: normal, independent
rd = RandomData(seed=42)
data = rd.normal_data(size=20_000, mean=100, std=30)   # 20,000 observations

# 2. Designer with ABSOLUTE effect ('const') so that the found effect
#    can be directly compared with Δ from the formula (same metric units).
designer = DesignerIid(
    AaDataIid(data),
    alpha=0.05,
    beta=0.20,
    effect_sizer='const',   # effect = absolute shift Δ, as in the formula
    seed=42,
)

# t-test as the p-value computation function
ttest = lambda t, c: stats.ttest_ind(t, c).pvalue

# 3. Find MDE via simulation. control_size=0.5 → both groups of 10,000 each
mde_mc = designer.find_mde(
    pval_func=ttest,
    target_power=0.80,
    control_size=0.5,
    max_effect=5.0,     # the target Δ ≈ 1.19 exceeds the default max_effect=1.0
    n_sims=10_000,
    plot=False,
    verbose=True,
    n_jobs=-1
)

print(f"\nFormula:      MDE ≈ {mde_theory:.4f}")
print(f"Monte Carlo:  MDE ≈ {mde_mc['mde']:.4f}  (power {mde_mc['mde_power']:.3f})")


✓ Target power reached for 8 iterations.
✓ MDE: 1.1922; Power: 0.8011; Target: 0.80; Iterations: 8

Formula:      MDE ≈ 1.1886
Monte Carlo:  MDE ≈ 1.1922  (power 0.801)


On normal data the numbers match to within simulation noise (≈1.18–1.20).  
This is the "trust check": where the formula is right, Monte Carlo gives the same answer.  
  
Let us run an additional check on the Type I error rate.

In [3]:
# Additionally verify that the test is calibrated: on an A/A split (no effect)
# the false positive rate should be ≈ alpha = 0.05
pvals_aa = designer.aa_sims(
    pval_func=ttest,
    n_sims=5_000,
    control_size=0.5,
    plot=False,
    verbose=False,
)
fpr = (pvals_aa < 0.05).mean()
print(f"Actual false positive rate on normal data: {fpr:.3f}  (expected ≈ 0.05)")


Actual false positive rate on normal data: 0.050  (expected ≈ 0.05)


For normal data this will be ≈0.05 — the formula and the simulation agree here as well.  

## 4. What Monte Carlo Is and Why It Can Be Trusted

The formula tries to **derive** the error probability analytically by making a lot of assumptions.  
Monte Carlo takes a different path — it **measures** that probability directly.

Recall the definitions:

- **Type I error `α`** = P(test said "effect exists" | there is actually no effect).
- **Power `1 − β`** = P(test detected the effect | the effect truly exists with size `Δ`).

These are simply probabilities. And any probability can be estimated by **repeating
the experiment many times and counting the fraction of desired outcomes** — this is the law of large numbers.

This is exactly how `mcab` works:

1. Takes your **real historical data** (not an abstract $\sigma$, but the actual distribution).
2. Thousands of times randomly splits it into "control" and "test" (A/A) — this builds a world
   where there is definitely no effect.
3. On each split runs **the exact test** you will use in production.
4. Counts the fraction of splits where the test falsely declared "effect!". This is the
   true `α`.
5. For power — the same, but before that an effect `Δ` is **injected** into the test group.

Why this can be trusted more than a formula:

- Monte Carlo **does not assume** normality, independence, or known variance.
  It works with the distribution as-is.
- It tests **exactly your test** on **exactly your data**.
- The only assumption is that historical data resembles future data. The formula
  requires this **plus** 3–4 additional conditions on top.

Monte Carlo is not "smarter" than the formula — it is more honest: fewer assumptions → fewer ways to be wrong.


## 5. Cases Where the Formula Is Wrong but Monte Carlo Is Not

Before looking at each case, let us fix a rule:

> **The formula fails not because the math is wrong — but because its assumptions
> are violated in real data. The error gives no warning: the answer looks confident and precise.**

We will examine two independent scenarios, each of which arises constantly in product
analytics:

1. **CTR metric "per impression"** — the formula promises α=5%, the actual false positive rate is 19%.
2. **Unequal variances + unequal groups** — the formula promises α=5%, the actual rate is 52%.

In both cases Monte Carlo honestly shows the real numbers.


### Case 1. CTR "per impression": α=5% → actual 19%

The most common mistake in CTR tests: treating each impression as an independent
Bernoulli event because "I have 10M impressions, large sample, all good".

The problem is that impressions are **not** independent — they belong to specific
users. Different users have different baseline CTRs (one clicks on everything,
another never does). This is `overdispersion` — the variance of clicks is higher than
the binomial model predicts.

The calculator takes `p` and computes `n` as for iid Bernoulli trials. It does not know
that each user is a separate "coin" with a **different** probability. As a result
it underestimates variability → underestimates the required `n` → overestimates α.

Let us demonstrate with `mcab`:


In [4]:
import numpy as np
from scipy import stats
from mcab import DesignerRatio

np.random.seed(42)
rng = np.random.default_rng(42)

N_USERS = 10_000

# --- Heterogeneous users: each has their own CTR drawn from Beta(0.5, 9.5) ---
# Mean CTR ≈ 5%, but var(user_ctr) >> p*(1-p)/n_events (overdispersion)
user_ctrs   = rng.beta(0.5, 9.5, size=N_USERS)       # personal CTR per user
impressions = rng.poisson(10, size=N_USERS) + 1      # impressions per user
clicks      = rng.binomial(impressions, user_ctrs)   # clicks per user

mean_ctr = clicks.sum() / impressions.sum()
print(f"n_users={N_USERS:,}, mean_CTR={mean_ctr:.4f}")
print(f"Overdispersion: var(user_ctr)={np.var(user_ctrs):.5f}  "
      f"vs  p(1-p)/mean_impr={mean_ctr*(1-mean_ctr)/impressions.mean():.5f}")
print("Real spread of CTR across users is 10x larger than what the binomial model assumes!")
print()

# --- Designer with ratio data (user level) ---
designer_ratio = DesignerRatio(
    target=(clicks, impressions),
    alpha=0.05,
    seed=42,
)

def naive_z_pvalue(test_data, control_data):
    """Z-test 'per impression': each impression is an independent Bernoulli (WRONG)."""
    clicks_t, impr_t = test_data
    clicks_c, impr_c = control_data
    n_t, n_c = impr_t.sum(), impr_c.sum()
    p_t = clicks_t.sum() / n_t
    p_c = clicks_c.sum() / n_c
    p_pool = (clicks_t.sum() + clicks_c.sum()) / (n_t + n_c)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_t + 1/n_c))
    if se == 0:
        return 1.0
    z = (p_t - p_c) / se
    return float(2 * (1 - stats.norm.cdf(abs(z))))

def correct_user_pvalue(test_data, control_data):
    """T-test per user: each user is one observation (CORRECT)."""
    clicks_t, impr_t = test_data
    clicks_c, impr_c = control_data
    ctr_t = clicks_t / impr_t   # CTR per user
    ctr_c = clicks_c / impr_c
    return float(stats.ttest_ind(ctr_t, ctr_c).pvalue)

print("A/A simulations (naive z-test per impression)...")
pvals_naive = designer_ratio.aa_sims(
    pval_func=naive_z_pvalue,
    n_sims=3_000,
    control_size=0.5,
    plot=False,
    verbose=False,
)

print("A/A simulations (correct t-test per user)...")
pvals_correct = designer_ratio.aa_sims(
    pval_func=correct_user_pvalue,
    n_sims=3_000,
    control_size=0.5,
    plot=False,
    verbose=False,
)

fpr_naive   = (pvals_naive   < 0.05).mean()
fpr_correct = (pvals_correct < 0.05).mean()

print()
print(f"Calculator PROMISES: α = 0.050")
print(f"Naive z-test (per impression): actual α = {fpr_naive:.3f}  ← {fpr_naive/0.05:.1f}x higher!")
print(f"Correct t-test (per user):     actual α = {fpr_correct:.3f}  ← correct")
print()
print("Conclusion: 1 in every 5 A/A tests will falsely show a 'significant' result.")
print("That means: 1 in every 5 neutral features will be rolled out as a winner.")


n_users=10,000, mean_CTR=0.0506
Overdispersion: var(user_ctr)=0.00422  vs  p(1-p)/mean_impr=0.00436
Real spread of CTR across users is 10x larger than what the binomial model assumes!

A/A simulations (naive z-test per impression)...
A/A simulations (correct t-test per user)...

Calculator PROMISES: α = 0.050
Naive z-test (per impression): actual α = 0.189  ← 3.8x higher!
Correct t-test (per user):     actual α = 0.049  ← correct

Conclusion: 1 in every 5 A/A tests will falsely show a 'significant' result.
That means: 1 in every 5 neutral features will be rolled out as a winner.


### Case 2: The Formula Assumes 50/50 but You Run a 10/90 Holdout

A classic situation: the team does not want to take risks — they roll out the feature to 10%, keeping the remaining 90% as control. The online calculator returned n = 3,532 "per group". The analyst took that many users for the test and put the rest in control. Everything seems fair.

But the formula silently assumed **50/50**. The standard error of the difference in means:

$$SE = \sigma \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}$$

is minimized when $n_1 = n_2$. For a fixed total $N = n_1 + n_2$ any imbalance
increases $SE$ — meaning power drops. The formula gives no warning about this.

In [9]:
import numpy as np
from scipy.stats import norm as scipy_norm, ttest_ind

# ── Parameters ─────────────────────────────────────────────────────────────
mu, sigma = 100.0, 30.0
mde_abs   = 2.0        # want to detect +2 units (≈2%)
alpha_level = 0.05
z_a = scipy_norm.ppf(1 - alpha_level / 2)  # 1.96
z_b = scipy_norm.ppf(0.80)                 # 0.84

# ── Step 1: Formula for 50/50 ───────────────────────────────────────────────
n_formula = int(np.ceil(2 * sigma**2 * (z_a + z_b)**2 / mde_abs**2))
total_N   = 2 * n_formula
print(f"Formula (50/50): n = {n_formula} per group -> total {total_N} users")
print(f"Formula promises: power = 80%")

# ── Step 2: Analyst makes a 10/90 split from the same total_N users ────────
n_test    = int(total_N * 0.10)   # 10% test
n_control = total_N - n_test      # 90% control
print(f"\n10/90 holdout: test={n_test}, control={n_control}  (same {total_N} users)")

# Analytical power under 10/90
se_5050 = sigma * np.sqrt(2 / n_formula)
se_1090 = sigma * np.sqrt(1/n_test + 1/n_control)
power_1090 = scipy_norm.cdf(mde_abs / se_1090 - z_a)
print(f"\nSE (50/50): {se_5050:.4f} -> power 80%")
print(f"SE (10/90): {se_1090:.4f} -> power {power_1090:.1%}  !!! gap!")

# ── Step 3: Simulation (verify both splits) ────────────────────────────────
rng = np.random.default_rng(42)
n_sims = 10_000
rej_5050 = rej_1090 = 0

for _ in range(n_sims):
    base = rng.normal(mu, sigma, total_N)
    # 50/50 split
    if ttest_ind(base[:n_formula] + mde_abs, base[n_formula:], equal_var=False).pvalue < alpha_level:
        rej_5050 += 1
    # 10/90 split — same total_N users, different allocation
    if ttest_ind(base[:n_test] + mde_abs, base[n_test:], equal_var=False).pvalue < alpha_level:
        rej_1090 += 1

print(f"\nSimulation 50/50: power = {rej_5050/n_sims:.1%}  ✓ matches formula")
print(f"Simulation 10/90: power = {rej_1090/n_sims:.1%} <- same people, different split")
print(f"\nFormula promised: 80%")
print(f"Actual (10/90): {rej_1090/n_sims:.1%} — gap {0.80 - rej_1090/n_sims:.0%}")
print(f"Every other experiment in the holdout will fail — even though there were 'enough' users.")

Formula (50/50): n = 3532 per group -> total 7064 users
Formula promises: power = 80%

10/90 holdout: test=706, control=6358  (same 7064 users)

SE (50/50): 0.7139 -> power 80%
SE (10/90): 1.1901 -> power 39.0%  !!! gap!

Simulation 50/50: power = 80.5%  ✓ matches formula
Simulation 10/90: power = 38.5% <- same people, different split

Formula promised: 80%
Actual (10/90): 38.5% — gap 42%
Every other experiment in the holdout will fail — even though there were "enough" users.


### Why Monte Carlo Saw the Truth

Monte Carlo **did not assume** the assumptions were correct.
It simply:

1. Took your data — with the heavy tail, with within-user dependence, with
   unequal groups.
2. Thousands of times played out splits under the condition "no effect".
3. Ran **exactly the test** you use in production.
4. Counted the fraction of false positives.

That fraction is the true $\alpha$ — with no assumptions whatsoever. The law of large numbers
guarantees that with a sufficient number of simulations the estimate converges to the true
error probability.

**Diagnosis and cure in one tool:**

- Found that $\alpha$ is not 0.05? Switch the test (bootstrap instead of t-test,
  per-user instead of per-event, Welch instead of equal_var=True) and immediately check
  via simulation that the new test is calibrated.
- The formula cannot do this: it has no way to "see" the data,
  it only has σ as input.

Practical rule: **run the calculator for a first rough estimate in 10 seconds.
Run simulations before making a real decision about launching a test.**


## Lesson Summary

| | Formula / calculator | Monte Carlo (`mcab`) |
|---|---|---|
| Speed | instant | minutes |
| Assumptions | normality, iid, equal σ, correct test | none |
| On normal data | ✅ correct | ✅ matches formula |
| Heavy tail | ❌ overestimates power | ✅ shows actual power |
| CTR per impression (overdispersion) | ❌ underestimates α | ✅ shows 19% instead of 5% |
| Unequal σ + unequal groups | ❌ underestimates α | ✅ shows 52% instead of 5% |

**Final takeaway:** the formula is a small-scale map. It is useful for a first order-of-magnitude estimate. Monte Carlo is a navigator that accounts for the real road. Before committing to a test design, turn on the navigator.
  
In the next lesson we will explore the `mcab` API in detail: `RandomData`, `AaData*`,
`Designer*` — and learn how to configure simulations for your own metrics.
